In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import gradio as gr
import chromadb

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents import Agent as OpenAIAgent, Runner, function_tool
from customized_agents.classifying_agent import ClassifyingAgent

load_dotenv(override=True)

In [ ]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collections = client.get_or_create_collection('products')

In [ ]:
classifyingAgent = ClassifyingAgent(collection=collections)


@function_tool
def classify_product(product_description: str) -> str:
    """Categorize a product from its description.

    Args:
        product_description: A description of the product that needs to be categorized.
    """
    return classifyingAgent.classify(product_description)


categorization_agent = OpenAIAgent(
    name="categorization_agent",
    instructions=(
        "You categorize products for an e-commerce platform. Use classify_product "
        "to determine the best high-level category. If the product is difficult "
        "to categorize, return the 3 categories it could belong to."
    ),
    tools=[classify_product],
)


In [ ]:


openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"


In [ ]:
@function_tool
def ping_manager(product_description: str, reason: str = ""):
    """Notify a store manager when a customer disputes a suggested category.

    Args:
        product_description: The product description or product being disputed.
        reason: Why the customer disagrees with the category, if provided.
    """
    print(f"Manager pinged for customer disagreement: {product_description}. Reason: {reason}")
    return "I understand you disagree with the suggested category. I have sent this product to a store manager for review."


tools = [
    categorization_agent.as_tool(
        tool_name="categorization_agent",
        tool_description=(
            "Categorize a product based on its description. If the product is "
            "difficult to categorize, return the 3 categories it could belong to."
        ),
    ),
    ping_manager,
]


In [ ]:
WELCOME_MESSAGE = "Hi! My name is Lana, and I'm here to help you register your product. First, please provide me with a description of your product. For example, what does it do, what are its features, and any other relevant information. Based on your description, I will sort your product into the correct category."
system_message ="""
You are a helpful assistant for an e-commerce platform that helps businesses register their products. Do not make up anything if you are unsure. Inform the customer when you contacted a store manager for further assistance. If the customer disagrees with the suggested category, call the ping_manager tool and tell them a store manager will review it. You will be provided with a product description, and your task is to categorize the product based on the description. If the product is difficult to categorize, you will return the 3 categories you think it could belong to.
"""

registration_agent = OpenAIAgent(
    name="Lana",
    instructions=system_message,
    model=MODEL,
    tools=tools,
)


In [ ]:

# from langchain_core import messages


chatbot = gr.Chatbot(
    type="messages",
    value=[
        {"role": "assistant", "content": WELCOME_MESSAGE}
    ]
)

def chat(message, history):
    messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": message})
    result = Runner.run_sync(registration_agent, input=messages)
    return result.final_output

gr.ChatInterface(fn=chat,chatbot=chatbot ,type="messages").launch()